## Run command for this worksheet

conda activate vk-llms
/opt/anaconda3/envs/vk-llms/bin/python "RAG Implementation/UdemyCourse/Worker.py"


# Initializing imports, keys, context

In [ ]:
# imports

import os
import glob
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI

# load environment variables
load_dotenv(override = True)
api_key = os.getenv("OPENAI_API_KEY")

# create openai client
openai = OpenAI(api_key=api_key)

MODEL = "gpt-4o-mini"

#initialize context

context = {}
# 
# Get the directory where the script is located
# script_dir = os.path.dirname(os.path.abspath(__file__))

# Load Employees Data
folder - knowledge-base/employees

In [ ]:
# employees_path = os.path.join(script_dir, "knowledge-base", "employees", "*")

# employees = glob.glob(employees_path)

employees = glob.glob("knowledge-base/employees/*")

for employee in employees:
    name = employee.split('/')[-1][:-3]
    doc = ""
    with open(employee, "r", encoding="utf-8") as f:
        doc = f.read()
    context[name]=doc



In [ ]:
print(context.keys())
# print("**")
# print(employees[0].split('/')[-1][:-3])

# Load Products Data
folder - knowledge-base/products

In [ ]:
products = glob.glob("knowledge-base/products/*")

for product in products:
    name = product.split('/')[-1][:-3]
    doc = ""
    with open(product, "r", encoding = "utf-8") as f:
        doc = f.read();
        context[name] = doc

In [ ]:
print(context["Carllm"])

# Building Context and System prompt

## System Message

In [ ]:


system_message = "You are expert in answering questions about Insurellm, the Insurance Tech company. Give brief, accurate answers. If you don't know the answer, say so. Do not make anything up if you haven't been provided with relevant context."



## Get Relevant Context

In [ ]:
def get_relevant_context(message) :
    relevant_context = []
    for context_title, context_details in context.items() :
        if context_title in message :
            relevant_context.append(context_details)
    return relevant_context

In [ ]:
print(get_relevant_context("Who is Alex Chen ?"))

## Add Relevant Context

In [ ]:
def add_context(message):
    relevant_context = get_relevant_context(message)
    if relevant_context:
        message += "\n\n The following additional context might be relevant in answering the question : \n\n"
        for relevant in relevant_context:
            message += relevant + "\n\n"
        return message
    else:
        # Return original message when no context is found
        return message

In [ ]:
# Won't print since it's not matching
print(add_context("Who is Alex Lancaseter ?"))

# This will fetch and add relevant Context
print(add_context("Who is Alex Chen ?"))

# Chat Message for Gradio

In [ ]:
def chat(message, history):

    print("MESSAGE \n")
    print(message)
    
    messages = [{"role" : "system", "content" : system_message}]
    if history :
        for user_message, assistant_message in history:
            messages.append({"role": "user", "content" : user_message})
            messages.append({"role": "assistant", "content" : assistant_message})
    # Add Context
    
    message = add_context(message)
    messages.append({"role" : "user", "content": message})

    print(messages)
    
    stream = openai.chat.completions.create(model = MODEL, messages = messages, stream = True)
    
    response = ""

    print(response)
    
    for chunk in stream : 
        response += chunk.choices[0].delta.content or ''
        yield response
    

# Gradio App Setup


In [ ]:
view = gr.ChatInterface(chat).launch()